In [1]:
import kagglehub
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import os

# Download latest version
# path = kagglehub.dataset_download("harunrai/digital-wallet-transactions")

# print("Path to dataset files:", path)

### Data processing with manipulation

In [2]:
# Import data from downloaded CSV file

df_digital = pd.read_csv("/Users/wildanhidayat/.cache/kagglehub/datasets/harunrai/digital-wallet-transactions/versions/1/digital_wallet_transactions.csv")

In [3]:
# Standardize column names
df_digital.columns = df_digital.columns.str.lower().str.replace(' ', '_')

In [4]:
# Simulate realistic skewed API latency and integration type by adding api_latency_ms
np.random.seed(42)

# Gamma distribution: shape(k)=2.0, scale(theta)=75 -> long right tail
# Base overhead = 80ms; base P50 ~ 180-230ms, tail can go up to 800ms or more
base_latency = np.random.gamma(shape=2.0, scale=75, size=(len(df_digital))) + 80

# Occasional 5% network or rail timeouts (adding 400-800ms)
timeout_spikes = np.where(
    np.random.rand(len(df_digital)) < 0.05, np.random.uniform(400, 800, size=(len(df_digital))), 0
)
df_digital["api_latency_ms"] = np.round(base_latency + timeout_spikes).astype(int)


In [5]:
# Simulate integration types to evaluate technical performance across different integration methods
integration_options = ["shopify_plugin", "raw_api", "hosted_page"]
integration_probs = [0.5, 0.35, 0.15]
df_digital["integration_type"] = np.random.choice(
    integration_options, size=len(df_digital), p=integration_probs
)

In [ ]:
# Calculate financial metrics
df_digital["transaction_date"] = pd.to_datetime(df_digital["transaction_date"])
df_digital["transaction_day"] = df_digital["transaction_date"].dt.date

# Successful transactions alias
df_digital["is_success"] = df_digital["transaction_status"].apply(
    lambda x: 1 if str(x).lower() == "successful" else 0)

# Calculate take rate % per transaction
df_digital["gpv_amount"] = df_digital["product_amount"].astype(float)
df_digital["gpv_amount"] = np.where(df_digital["is_success"] == 1, df_digital["gpv_amount"], 0.0)

df_digital["gross_revenue"] = df_digital["transaction_fee"].astype(float)
df_digital["gross_revenue"] = np.where(df_digital["is_success"] == 1, df_digital["gross_revenue"], 0.0)

df_digital["take_rate"] = np.where(
    df_digital["gpv_amount"] > 0,
    np.round(df_digital["gross_revenue"] / df_digital["gpv_amount"], 4),
    0.0
)

### Data Mart 1: Finance

In [ ]:
# Aggregate financial metrics by day, product category, and payment method
mart_finance = df_digital.groupby(['transaction_day', 'product_category', 'payment_method']).agg(
    total_transactions = ('transaction_id', 'count'),
    successful_transactions = ('is_success', 'sum'),
    total_gpv = ('gpv_amount', 'sum'),
    net_revenue = ('gross_revenue', 'sum'),
    avg_take_rate = ('take_rate', 'mean')
).reset_index()

In [ ]:
# Calculate auth success rate
mart_finance["auth_success_rate"] = np.round(
    mart_finance["successful_transactions"] / mart_finance["total_transactions"], 4
)

### Data Mart 2: System

In [19]:
# Aggregate API latency metrics by day, product category, and payment method
mart_system = df_digital.groupby(['transaction_day', 'integration_type', 'payment_method']).agg(
        total_api_requests = ('transaction_id', 'count'),
        failed_requests = ('is_success', lambda x: (1 - x).sum()),
        p50_latency_ms = ('api_latency_ms', 'median'),
        p95_latency_ms = ('api_latency_ms', lambda x: np.percentile(x, 95)),
        p99_latency_ms = ('api_latency_ms', lambda x: np.percentile(x, 99))
    ).reset_index()

In [20]:
# Calculate the technical error rate
mart_system['technical_error_rate'] = np.round(
    mart_system['failed_requests'] / mart_system['total_api_requests'], 4
)

### Push Data to the Database

In [29]:
# Database connection
DB_USER = "myuser"
DB_PASSWORD = "mypassword"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "digital_transactions_db"

In [30]:
# Engine connection to the database
engine = create_engine(f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [32]:
with engine.connect() as conn:
    # General table
    df_digital.to_sql(
        "digital_transactions",
        con = conn,
        if_exists = "replace",
        index = False
    )

    # Mart finance
    mart_finance.to_sql(
        "digital_mart_finance",
        con = conn,
        if_exists = "replace",
        index = False
    )

    # Mart system
    mart_system.to_sql(
        "digital_mart_system",
        con = conn,
        if_exists = "replace",
        index = False
    )